# HMA — Baseline Capture (v1.0)
## NH-65 Hyderabad–Vijayawada, pre-works snapshot

**The deadline.** NHAI set a bid deadline of 11 November 2026 for the six-laning of the
181 km Hyderabad–Vijayawada stretch of NH-65, with works reported to begin in **January
2027**. Once construction starts, OSM mappers update the database to match the new
reality and the pre-works state disappears from the live map. This notebook captures it
while it still exists.

**What makes this worth doing tonight.** The perishable asset is not our metrics — it is
the raw dated extract. Archive that with proper provenance and we can compute *any*
metric against it years later, including metrics we have not thought of yet and
including a corrected Detour Index if the current one is still wrong. Miss the window
and no amount of later cleverness recovers it.

**What this produces**
1. A clipped `.osm.pbf` of the corridor, archived to Drive.
2. A SHA-256 checksum, so we can prove years later that the file is unaltered.
3. A provenance record: source URL, download date, OSM replication timestamp from the
   file header, osmium version, bounding box, byte count.
4. A pre-works metrics summary — the "before" numbers.

**This is an archive, not an analysis.** It makes no claim about the road. It only
establishes, verifiably, what the open map said on a given date.

**Repeat after works complete** with the same notebook and an unchanged corridor
definition. Same ruler, both ends.

## 1. Install

In [ ]:
!pip -q install osmium tabulate
!apt-get -qq install -y osmium-tool > /dev/null 2>&1
import osmium, pandas, subprocess
print("osmium-tool:", subprocess.run(["osmium","--version"],
      capture_output=True, text=True).stdout.split("\n")[0])

## 2. Corridor definition — FREEZE THIS

These bounds define what "the corridor" means for every future comparison. Once the
first baseline is archived, changing them invalidates the before/after pair. If they
ever need to change, make a new corridor with a new name rather than editing this one.

In [ ]:
CORRIDOR_ID   = "NH65_HYD_VJA"
CORRIDOR_NAME = "NH-65 Hyderabad to Vijayawada (six-laning, NHAI)"
CORRIDOR_NOTE = ("Pre-works baseline. NHAI bid deadline 11 Nov 2026; "
                 "works reported to begin Jan 2027. Source: news reporting, "
                 "not an official NHAI document.")

# Bounding box spanning Hyderabad to Vijayawada along NH-65.
# (min_lon, min_lat, max_lon, max_lat)
CORRIDOR_BBOX = (78.30, 16.30, 80.75, 17.60)

EXTRACT_URL = "https://download.geofabrik.de/asia/india/southern-zone-latest.osm.pbf"
PBF_PATH    = "/content/southern-zone.osm.pbf"
BASE_DIR    = "/content/hma_baseline"

import os
os.makedirs(BASE_DIR, exist_ok=True)
print(CORRIDOR_ID, CORRIDOR_BBOX)

## 3. Download the source extract

In [ ]:
import subprocess, os
if not (os.path.exists(PBF_PATH) and os.path.getsize(PBF_PATH) > 1_000_000):
    print("downloading", EXTRACT_URL)
    if subprocess.call(["wget","-q","--show-progress","-O",PBF_PATH,EXTRACT_URL]) != 0:
        raise SystemExit("Download failed — check EXTRACT_URL.")
print(round(os.path.getsize(PBF_PATH)/1e6,1), "MB source")

## 4. Read the source file's own replication timestamp

This is the single most important provenance field. It is OSM's statement of how
current the data is — independent of when we happened to download it.

In [ ]:
info = subprocess.run(["osmium","fileinfo","-e",PBF_PATH],
                      capture_output=True, text=True).stdout
src_timestamp = None
for line in info.split("\n"):
    s = line.strip()
    if s.startswith("osmosis_replication_timestamp="):
        src_timestamp = s.split("=",1)[1].strip()
    elif s.lower().startswith("timestamp:") and src_timestamp is None:
        src_timestamp = s.split(":",1)[1].strip()

print("OSM replication timestamp:", src_timestamp or "NOT FOUND")
if not src_timestamp:
    print("WARNING: no replication timestamp. Provenance is weaker — record the "
          "download date and note the gap.")

## 5. Clip the corridor

In [ ]:
x0, y0, x1, y1 = CORRIDOR_BBOX
CLIP = f"{BASE_DIR}/{CORRIDOR_ID}_baseline.osm.pbf"

r = subprocess.run(["osmium","extract","-b",f"{x0},{y0},{x1},{y1}",
                    PBF_PATH,"-o",CLIP,"--overwrite","-s","complete_ways"],
                   capture_output=True, text=True)
if r.returncode != 0:
    print(r.stderr[-2000:]); raise SystemExit("Clip failed.")

print(round(os.path.getsize(CLIP)/1e6,1), "MB corridor extract")

## 6. Checksum and provenance record

The checksum is what lets anyone — including a sceptical official in 2029 — verify that
the file we publish our 'before' numbers from is byte-identical to the one we archived
tonight.

In [ ]:
import hashlib, datetime, json

def sha256_of(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

digest = sha256_of(CLIP)
captured_at = datetime.datetime.now(datetime.timezone.utc).isoformat()

provenance = {
    "corridor_id": CORRIDOR_ID,
    "corridor_name": CORRIDOR_NAME,
    "note": CORRIDOR_NOTE,
    "bbox_min_lon_min_lat_max_lon_max_lat": list(CORRIDOR_BBOX),
    "source_url": EXTRACT_URL,
    "osm_replication_timestamp": src_timestamp,
    "captured_at_utc": captured_at,
    "file_name": os.path.basename(CLIP),
    "file_bytes": os.path.getsize(CLIP),
    "sha256": digest,
    "osmium_tool_version": subprocess.run(["osmium","--version"],
        capture_output=True, text=True).stdout.split("\n")[0],
    "licence": "Map data (c) OpenStreetMap contributors, ODbL 1.0",
    "capture_stage": "pre-works",
    "produced_by": "HMA Baseline Capture v1.0",
}

with open(f"{BASE_DIR}/{CORRIDOR_ID}_provenance.json","w") as f:
    json.dump(provenance, f, indent=2)

print(json.dumps(provenance, indent=2))

## 7. Pre-works metrics — the 'before' numbers

Deliberately the same measurements as the Phase 1 coverage audit, so the after-capture
is directly comparable. These describe what OSM records, not what exists on the ground.

In [ ]:
import osmium, collections, math

HIGHWAY_MAIN = {"motorway","trunk","primary","secondary","tertiary",
                "motorway_link","trunk_link","primary_link","secondary_link",
                "tertiary_link","unclassified","residential","living_street"}
ARTERIAL = {"motorway","trunk","primary","secondary"}
MILITARY_KEYS = {"military","military_service"}

def haversine_m(lon1, lat1, lon2, lat2):
    R = 6371008.8
    p1, p2 = math.radians(lat1), math.radians(lat2)
    a = (math.sin((p2-p1)/2)**2
         + math.cos(p1)*math.cos(p2)*math.sin(math.radians(lon2-lon1)/2)**2)
    return 2*R*math.asin(math.sqrt(a))

class BaselinePass(osmium.SimpleHandler):
    def __init__(self):
        super().__init__()
        self.km = collections.Counter()
        self.arterial_divided_m = 0.0
        self.arterial_m = 0.0
        self.lanes_tagged = 0
        self.ways_counted = 0
        self.nodes = collections.Counter()
        self.restrictions = collections.Counter()
        self.military_stripped = 0

    def way(self, w):
        tags = dict(w.tags)
        hw = tags.get("highway")
        if hw not in HIGHWAY_MAIN:
            return
        if any(k in MILITARY_KEYS for k in tags) or tags.get("landuse") == "military":
            self.military_stripped += 1
            tags = {k: v for k, v in tags.items()
                    if k not in MILITARY_KEYS and k not in ("landuse","name","operator")}
        try:
            length = osmium.geom.haversine_distance(w.nodes)
        except Exception:
            return
        self.km[hw] += length / 1000.0
        self.ways_counted += 1
        if tags.get("lanes"):
            self.lanes_tagged += 1
        if hw in ARTERIAL:
            self.arterial_m += length
            if tags.get("oneway") in ("yes","true","1","-1") or tags.get("dual_carriageway") == "yes":
                self.arterial_divided_m += length

    def node(self, n):
        t = dict(n.tags)
        if not t:
            return
        if t.get("highway") == "crossing":
            self.nodes["crossing"] += 1
        elif t.get("highway") == "traffic_signals":
            self.nodes["traffic_signals"] += 1

    def relation(self, r):
        t = dict(r.tags)
        if t.get("type") == "restriction":
            self.restrictions[t.get("restriction") or "unspecified"] += 1

b = BaselinePass()
b.apply_file(CLIP, locations=True)

total_km = sum(b.km.values())
art_km = b.arterial_m / 1000.0
uturn = sum(v for k, v in b.restrictions.items() if "u_turn" in k)

baseline = {
    "corridor_id": CORRIDOR_ID,
    "capture_stage": "pre-works",
    "osm_replication_timestamp": src_timestamp,
    "road_km_total": round(total_km, 1),
    "arterial_km": round(art_km, 1),
    "arterial_divided_pct": round(100*b.arterial_divided_m/b.arterial_m, 1) if b.arterial_m else 0.0,
    "lanes_tagged_pct": round(100*b.lanes_tagged/b.ways_counted, 1) if b.ways_counted else 0.0,
    "crossings": b.nodes["crossing"],
    "crossings_per_arterial_km": round(b.nodes["crossing"]/art_km, 3) if art_km else 0.0,
    "traffic_signals": b.nodes["traffic_signals"],
    "turn_restrictions_total": sum(b.restrictions.values()),
    "u_turn_restrictions": uturn,
    "military_attributes_stripped": b.military_stripped,
}
for k, v in baseline.items():
    print(f"  {k:32s} {v}")

## 8. Tests

In [ ]:
# provenance completeness
for key in ["corridor_id","sha256","captured_at_utc","file_bytes","source_url","licence"]:
    assert provenance.get(key), f"provenance missing {key}"
assert len(provenance["sha256"]) == 64, "sha256 must be 64 hex chars"

# checksum must be reproducible
assert sha256_of(CLIP) == digest, "checksum not stable — file changed under us"

# the extract must actually contain road data
assert baseline["road_km_total"] > 100, "corridor extract looks empty"
assert baseline["arterial_km"] > 10, "no arterial road found — check CORRIDOR_BBOX"

# haversine sanity
assert abs(haversine_m(78.4, 17.4, 78.4, 17.4009) - 100) < 3
print("PASS — archive is complete, checksummed and non-empty")

## 9. Write the baseline record and push to Drive

In [ ]:
import shutil, pandas as pd

with open(f"{BASE_DIR}/{CORRIDOR_ID}_baseline_metrics.json","w") as f:
    json.dump(baseline, f, indent=2)
pd.DataFrame([baseline]).to_csv(f"{BASE_DIR}/{CORRIDOR_ID}_baseline_metrics.csv", index=False)

with open(f"{BASE_DIR}/README_{CORRIDOR_ID}.md","w") as f:
    f.write(f"# HMA Baseline — {CORRIDOR_NAME}\n\n")
    f.write(f"**Capture stage:** pre-works\n\n")
    f.write(f"**OSM replication timestamp:** {src_timestamp}\n\n")
    f.write(f"**Captured (UTC):** {captured_at}\n\n")
    f.write(f"**SHA-256:** `{digest}`\n\n")
    f.write(f"**Bounding box:** {CORRIDOR_BBOX}\n\n")
    f.write(f"{CORRIDOR_NOTE}\n\n")
    f.write("Map data (c) OpenStreetMap contributors, ODbL 1.0. "
            "HMA derived metrics published under ODbL.\n\n")
    f.write("## Pre-works metrics\n\n")
    f.write(pd.DataFrame([baseline]).T.rename(columns={0:"value"}).to_markdown())
    f.write("\n\n## How to repeat this after works complete\n\n")
    f.write("Run this notebook unchanged. Do NOT edit CORRIDOR_BBOX — altering it "
            "invalidates the before/after comparison. Set capture_stage to "
            "'post-works' and archive alongside this file.\n\n")
    f.write("These figures describe what OpenStreetMap recorded on the date above. "
            "They are not a survey of the road.\n")

from google.colab import drive
drive.mount("/content/drive")
dest = f"/content/drive/MyDrive/Anthropic/Hyderabad Mobility Atlas/baselines/{CORRIDOR_ID}"
os.makedirs(dest, exist_ok=True)
for fn in os.listdir(BASE_DIR):
    shutil.copy(f"{BASE_DIR}/{fn}", f"{dest}/{fn}")
print("archived to Drive:", dest)
for fn in sorted(os.listdir(dest)):
    print(f"  {fn}  ({os.path.getsize(f'{dest}/{fn}')/1e6:.1f} MB)")

## 10. What has and has not been established

**Established:** what OpenStreetMap recorded for this corridor on a specific, verifiable
date, archived with a checksum so the claim can be audited later.

**Not established:** anything about the road itself. OSM coverage along this corridor has
not been audited, and the Phase 1 audit found Hyderabad's coverage thin. The before/after
comparison will be a comparison of *recorded* geometry, and the gap between recorded and
real must be stated every time these numbers are used.

**Also worth capturing before January:** the corridor's current satellite imagery date,
and any official NHAI alignment drawing obtainable by RTI. Neither is in this notebook.